In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

df = pd.read_csv("metrics_all_LLMs.csv")

# Show Q2 rows for manual verification for kimi-k2.5 model
df[(df["model"] == "moonshotai/kimi-k2.5") & (df["question_number"] == "Q2")]



,model,question_number,conclusion,metric,precision,recall,f1,gold_size,pred_size
84,moonshotai/kimi-k2.5,Q2,C1,rules_selected,1.000000,0.727273,0.842105,11,8
85,moonshotai/kimi-k2.5,Q2,C1,edges_support,0.600000,0.562500,0.580645,16,15
86,moonshotai/kimi-k2.5,Q2,C2,rules_selected,1.000000,0.727273,0.842105,11,8
87,moonshotai/kimi-k2.5,Q2,C2,edges_support,0.466667,0.636364,0.538462,11,15


In [2]:

df.groupby(["model", "metric"])[["precision", "recall", "f1"]].mean().round(3).reset_index().sort_values(["metric", "f1"], ascending=[True, False])



,model,metric,precision,recall,f1
6,moonshotai/kimi-k2.5,edges_support,0.519,0.474,0.479
0,anthropic/claude-opus-4.6,edges_support,0.589,0.374,0.432
8,openai/gpt-5.4,edges_support,0.501,0.402,0.426
2,google/gemini-2.5-pro,edges_support,0.521,0.332,0.373
4,google/gemini-3.1-pro-preview,edges_support,0.646,0.235,0.326
7,moonshotai/kimi-k2.5,rules_selected,0.903,0.667,0.756
9,openai/gpt-5.4,rules_selected,0.895,0.593,0.694
1,anthropic/claude-opus-4.6,rules_selected,0.889,0.562,0.672
3,google/gemini-2.5-pro,rules_selected,0.893,0.487,0.597
5,google/gemini-3.1-pro-preview,rules_selected,0.900,0.345,0.490


In [3]:
# Table 4: Per-question results for Kimi K2.5 (best-performing model)
kimi = df[df["model"] == "moonshotai/kimi-k2.5"].copy()

rules = kimi[kimi["metric"] == "rules_selected"][["question_number", "conclusion", "precision", "recall", "f1"]].rename(
    columns={"precision": ("Rule selection (R)", "P"), "recall": ("Rule selection (R)", "R"), "f1": ("Rule selection (R)", "F1")}
)
edges = kimi[kimi["metric"] == "edges_support"][["question_number", "conclusion", "precision", "recall", "f1"]].rename(
    columns={"precision": ("Support Edge prediction (A)", "P"), "recall": ("Support Edge prediction (A)", "R"), "f1": ("Support Edge prediction (A)", "F1")}
)

merged = rules.merge(edges, on=["question_number", "conclusion"])
merged.columns = pd.MultiIndex.from_tuples(
    [("Q", ""), ("C", "")] + list(merged.columns[2:])
)
merged = merged.sort_values([("Q", ""), ("C", "")]).reset_index(drop=True)
merged.iloc[:, 2:] = merged.iloc[:, 2:].astype(float).round(3)
merged


Q   C Rule selection (R)               Support Edge prediction (A)         \
                           P      R     F1                           P      R   
0  Q1  C1              1.000  0.833  0.909                       0.556  0.500   
1  Q1  C2              0.800  0.800  0.800                       0.500  0.800   
2  Q2  C1              1.000  0.727  0.842                       0.600  0.562   
3  Q2  C2              1.000  0.727  0.842                       0.467  0.636   
4  Q3  C1              1.000  0.600  0.750                       0.562  0.346   
5  Q3  C2              0.857  0.857  0.857                       0.750  0.818   
6  Q4  C1              0.875  0.389  0.538                       0.538  0.269   
7  Q4  C2              0.500  0.455  0.476                       0.357  0.312   
8  Q5  C1              1.000  0.786  0.880                       0.190  0.167   
9  Q5  C2              1.000  0.500  0.667                       0.667  0.333   

          
      F1  
0  0.526  
1  0.615  
2  0.581  
3  0.538  
4  0.429  
5  0.783  
6  0.359  
7  0.333  
8  0.178  
9  0.444